Cell 1 – Khởi tạo và đọc dữ liệu

In [3]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import warnings
import os

warnings.filterwarnings("ignore")

CURRENT_DIR = Path.cwd().resolve()
candidates = [CURRENT_DIR, *CURRENT_DIR.parents]

for env_name in ("VSCODE_CWD", "GITHUB_WORKSPACE"):
    env_dir = os.environ.get(env_name)
    if env_dir:
        candidates.append(Path(env_dir).resolve())

PROJECT_DIR = next(
    (candidate for candidate in candidates if (candidate / "data" / "raw").is_dir()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục project chứa data/raw/. "
        "Hãy mở VS Code tại thư mục SKU-Weekly-Sales-Forecasting "
        "hoặc đặt kernel working directory về thư mục project. "
        f"Thư mục hiện tại: {CURRENT_DIR}"
    )

RAW_DIR = PROJECT_DIR / "data" / "raw"
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_safely(file_path):
    for encoding in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )
        except Exception:
            pass
    raise ValueError(f"Không đọc được {file_path}")


tables = {
    file.stem: read_csv_safely(file)
    for file in sorted(RAW_DIR.glob("*.csv"))
}

if not tables:
    raise FileNotFoundError("Không tìm thấy dữ liệu CSV trong data/raw/.")

# Mốc tham chiếu cho đánh giá Timeliness.
# Có thể thay bằng ngày chốt dữ liệu được nhóm thống nhất.
REFERENCE_DATE = pd.Timestamp.today().normalize()

print("Số bảng:", len(tables))
print("Reference date:", REFERENCE_DATE.date())

Số bảng: 27
Reference date: 2026-11-25


Cell 2 – Khai báo quy tắc và hàm hỗ trợ

In [4]:
def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def find_column(df, candidates):
    normalized = {
        normalize_name(col): col
        for col in df.columns
    }

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]

    for candidate in candidates:
        key = normalize_name(candidate)

        if len(key) < 3:
            continue

        for norm_col, original_col in normalized.items():
            if key in norm_col:
                return original_col

    return None


CANDIDATES = {
    "date": [
        "OrderDate", "Order_Date", "SalesDate",
        "TransactionDate", "CreatedAt", "Date"
    ],
    "quantity": [
        "Quantity", "Qty", "SalesQuantity", "UnitsSold"
    ],
    "price": [
        "UnitPrice", "Price", "SellingPrice",
        "UnitCostAtSale"
    ],
    "sku": [
        "SKU", "SKUCode", "ProductSKU", "VariantSKU"
    ],
    "status": [
        "OrderStatus", "Status", "Order_State"
    ]
}

# Các tên cột có khả năng là khóa định danh.
KEY_NAME_HINTS = [
    "id", "_id", "code", "sku", "number"
]

def is_key_candidate(column):
    name = str(column).lower()
    return (
        name == "id"
        or name.endswith("_id")
        or name.endswith("code")
        or name in ["sku", "ordernumber", "orderno"]
    )

Cell 3 – Completeness, Validity, Uniqueness và Accuracy

In [5]:
dimension_rows = []
issue_rows = []

for table_name, df in tables.items():

    total_cells = int(df.shape[0] * df.shape[1])
    missing_cells = int(df.isna().sum().sum())

    completeness_score = (
        1 - missing_cells / total_cells
        if total_cells > 0 else np.nan
    )

    # Validity: chỉ tính trên các giá trị có mặt và kiểm tra được.
    valid_checks = []
    valid_passed = 0
    valid_total = 0

    qty_col = find_column(df, CANDIDATES["quantity"])
    price_col = find_column(df, CANDIDATES["price"])

    if qty_col:
        qty = pd.to_numeric(df[qty_col], errors="coerce")
        present = qty.notna()

        # Quantity âm được ghi nhận là cần điều tra,
        # không kết luận ngay là lỗi vì có thể là hàng trả/điều chỉnh.
        passed = qty[present].notna() & np.isfinite(qty[present])

        valid_passed += int(passed.sum())
        valid_total += int(present.sum())

        negative_count = int((qty < 0).sum())

        if negative_count:
            issue_rows.append({
                "table": table_name,
                "dimension": "Accuracy / Validity",
                "column": qty_col,
                "issue": "Quantity âm cần điều tra",
                "affected_rows": negative_count,
                "recommended_action": (
                    "Đối soát nghiệp vụ: hàng trả, điều chỉnh hay lỗi nhập liệu"
                )
            })

    if price_col:
        price = pd.to_numeric(df[price_col], errors="coerce")
        present = price.notna()

        passed = (
            price[present].notna()
            & np.isfinite(price[present])
            & (price[present] >= 0)
        )

        valid_passed += int(passed.sum())
        valid_total += int(present.sum())

        invalid_price_count = int((price < 0).sum())

        if invalid_price_count:
            issue_rows.append({
                "table": table_name,
                "dimension": "Validity",
                "column": price_col,
                "issue": "Giá trị giá âm",
                "affected_rows": invalid_price_count,
                "recommended_action": "Kiểm tra quy tắc giá và đơn vị tiền tệ"
            })

    validity_score = (
        valid_passed / valid_total
        if valid_total > 0 else np.nan
    )

    duplicate_rows = int(df.duplicated().sum())
    uniqueness_score = (
        1 - duplicate_rows / len(df)
        if len(df) > 0 else np.nan
    )

    # Accuracy cần nguồn tham chiếu; không có reference thì để N/A.
    accuracy_score = np.nan

    dimension_rows.extend([
        {
            "table": table_name,
            "dimension": "Completeness",
            "score": completeness_score,
            "numerator": total_cells - missing_cells,
            "denominator": total_cells,
            "note": "Tỷ lệ ô không thiếu"
        },
        {
            "table": table_name,
            "dimension": "Validity",
            "score": validity_score,
            "numerator": valid_passed,
            "denominator": valid_total,
            "note": "Chỉ gồm quy tắc đã kiểm tra được"
        },
        {
            "table": table_name,
            "dimension": "Uniqueness",
            "score": uniqueness_score,
            "numerator": len(df) - duplicate_rows,
            "denominator": len(df),
            "note": "Dựa trên dòng trùng hoàn toàn"
        },
        {
            "table": table_name,
            "dimension": "Accuracy",
            "score": accuracy_score,
            "numerator": np.nan,
            "denominator": np.nan,
            "note": "N/A: chưa có dữ liệu tham chiếu để đối soát"
        }
    ])

    if missing_cells:
        issue_rows.append({
            "table": table_name,
            "dimension": "Completeness",
            "column": "*",
            "issue": "Có giá trị thiếu",
            "affected_rows": missing_cells,
            "recommended_action": (
                "Phân tích mức độ quan trọng của trường trước khi xử lý"
            )
        })

    if duplicate_rows:
        issue_rows.append({
            "table": table_name,
            "dimension": "Uniqueness",
            "column": "*",
            "issue": "Có dòng trùng hoàn toàn",
            "affected_rows": duplicate_rows,
            "recommended_action": (
                "Kiểm tra khóa và xác định trùng thực sự trước khi loại"
            )
        })


dimension_df = pd.DataFrame(dimension_rows)
issues_df = pd.DataFrame(issue_rows)

display(dimension_df)
display(issues_df)

,table,dimension,score,numerator,denominator,note
0,Brands,Completeness,1.0,132.0,132.0,Tỷ lệ ô không thiếu
1,Brands,Validity,NaN,0.0,0.0,Chỉ gồm quy tắc đã kiểm tra được
2,Brands,Uniqueness,1.0,66.0,66.0,Dựa trên dòng trùng hoàn toàn
3,Brands,Accuracy,NaN,NaN,NaN,N/A: chưa có dữ liệu tham chiếu để đối soát
4,BrandSubcategoryMap,Completeness,1.0,620.0,620.0,Tỷ lệ ô không thiếu
...,...,...,...,...,...,...
103,Subcategories,Accuracy,NaN,NaN,NaN,N/A: chưa có dữ liệu tham chiếu để đối soát
104,Suppliers,Completeness,1.0,64.0,64.0,Tỷ lệ ô không thiếu
105,Suppliers,Validity,NaN,0.0,0.0,Chỉ gồm quy tắc đã kiểm tra được
106,Suppliers,Uniqueness,1.0,16.0,16.0,Dựa trên dòng trùng hoàn toàn


,table,dimension,column,issue,affected_rows,recommended_action
0,CustomerReturns,Completeness,*,Có giá trị thiếu,32036,Phân tích mức độ quan trọng của trường trước k...
1,CustomerReturns,Uniqueness,*,Có dòng trùng hoàn toàn,25,Kiểm tra khóa và xác định trùng thực sự trước ...
2,Customers,Completeness,*,Có giá trị thiếu,1105,Phân tích mức độ quan trọng của trường trước k...
3,Customers,Uniqueness,*,Có dòng trùng hoàn toàn,368,Kiểm tra khóa và xác định trùng thực sự trước ...
4,InventoryBalances,Completeness,*,Có giá trị thiếu,419,Phân tích mức độ quan trọng của trường trước k...
5,InventoryMovements,Completeness,*,Có giá trị thiếu,5474228,Phân tích mức độ quan trọng của trường trước k...
6,Locations,Completeness,*,Có giá trị thiếu,26,Phân tích mức độ quan trọng của trường trước k...
7,Payments,Completeness,*,Có giá trị thiếu,506086,Phân tích mức độ quan trọng của trường trước k...
8,Products,Completeness,*,Có giá trị thiếu,2148,Phân tích mức độ quan trọng của trường trước k...
9,Products,Uniqueness,*,Có dòng trùng hoàn toàn,1,Kiểm tra khóa và xác định trùng thực sự trước ...


Cell 4 – Timeliness: ngày tương lai và khoảng thời gian khảo sát

In [6]:
timeliness_rows = []

for table_name, df in tables.items():

    date_col = find_column(df, CANDIDATES["date"])

    if not date_col:
        timeliness_rows.append({
            "table": table_name,
            "date_column": None,
            "min_date": None,
            "max_date": None,
            "future_records": np.nan,
            "timeliness_score": np.nan,
            "note": "Không tìm thấy cột ngày phù hợp"
        })
        continue

    dates = pd.to_datetime(df[date_col], errors="coerce")
    valid_dates = dates.dropna()

    if valid_dates.empty:
        timeliness_rows.append({
            "table": table_name,
            "date_column": date_col,
            "min_date": None,
            "max_date": None,
            "future_records": np.nan,
            "timeliness_score": np.nan,
            "note": "Không chuyển đổi được giá trị ngày"
        })
        continue

    future_count = int((valid_dates > REFERENCE_DATE).sum())
    timeliness_score = 1 - future_count / len(valid_dates)

    timeliness_rows.append({
        "table": table_name,
        "date_column": date_col,
        "min_date": valid_dates.min(),
        "max_date": valid_dates.max(),
        "future_records": future_count,
        "timeliness_score": timeliness_score,
        "note": "Tỷ lệ ngày không vượt quá Reference Date"
    })

    if future_count:
        issues_df.loc[len(issues_df)] = {
            "table": table_name,
            "dimension": "Timeliness",
            "column": date_col,
            "issue": "Có ngày lớn hơn Reference Date",
            "affected_rows": future_count,
            "recommended_action": "Kiểm tra ngày giao dịch và ngày hệ thống"
        }

timeliness_df = pd.DataFrame(timeliness_rows)
display(timeliness_df)

,table,date_column,min_date,max_date,future_records,timeliness_score,note
0,Brands,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
1,BrandSubcategoryMap,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
2,Categories,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
3,CustomerReturns,ReturnDate,2023-07-02,2026-12-20 00:00:00,1.0,0.999958,Tỷ lệ ngày không vượt quá Reference Date
4,Customers,CustomerSinceDate,2019-01-01,2025-12-31 00:00:00,0.0,1.000000,Tỷ lệ ngày không vượt quá Reference Date
5,Governorates,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
6,InventoryBalances,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
7,InventoryMovements,MovementDateTime,2023-07-01,2025-12-31 18:00:00,0.0,1.000000,Tỷ lệ ngày không vượt quá Reference Date
8,Locations,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp
9,PaymentMethods,None,NaT,NaT,NaN,NaN,Không tìm thấy cột ngày phù hợp


Cell 5 – Consistency: khóa ứng viên, giá trị trạng thái và khóa ngoại

In [7]:
consistency_rows = []

# Kiểm tra trạng thái có nhiều cách viết gần giống nhau.
for table_name, df in tables.items():
    status_col = find_column(df, CANDIDATES["status"])

    if status_col:
        status_values = (
            df[status_col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        status_normalized = status_values.str.lower()

        consistency_rows.append({
            "table": table_name,
            "check": "Status values",
            "column": status_col,
            "result": " | ".join(
                status_values.value_counts().head(30).index.tolist()
            ),
            "issue_count": int(
                status_normalized.nunique() - status_values.nunique()
            ),
            "note": "Cần rà soát các cách viết trạng thái"
        })

# Kiểm tra cột cùng tên giữa các bảng và tỷ lệ khớp khóa.
for i, table_a in enumerate(tables):
    for table_b in list(tables.keys())[i + 1:]:

        df_a = tables[table_a]
        df_b = tables[table_b]

        cols_a = {
            normalize_name(c): c for c in df_a.columns
        }
        cols_b = {
            normalize_name(c): c for c in df_b.columns
        }

        for key in set(cols_a) & set(cols_b):
            col_a = cols_a[key]
            col_b = cols_b[key]

            if not is_key_candidate(col_a):
                continue

            left_values = set(
                df_a[col_a].dropna().astype(str)
            )
            right_values = set(
                df_b[col_b].dropna().astype(str)
            )

            if not left_values:
                continue

            unmatched = left_values - right_values

            consistency_rows.append({
                "table": f"{table_a} -> {table_b}",
                "check": "Shared key coverage",
                "column": col_a,
                "result": f"{len(unmatched)} giá trị không tìm thấy ở bảng phải",
                "issue_count": len(unmatched),
                "note": (
                    f"Khóa đối chiếu: {col_a} = {col_b}; "
                    "cần xác nhận quan hệ nghiệp vụ"
                )
            })

consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)

,table,check,column,result,issue_count,note
0,CustomerReturns,Status values,RefundStatus,Completed,0,Cần rà soát các cách viết trạng thái
1,PurchaseOrders,Status values,PurchaseOrderStatus,Closed | Close,0,Cần rà soát các cách viết trạng thái
2,SalesOrders,Status values,OrderStatus,Completed | Cancelled | Complete | Canceled,0,Cần rà soát các cách viết trạng thái
3,Shipments,Status values,FinalStatusDate,2025-12-30 | 2025-12-31 | 2025-11-20 | 2025-11...,0,Cần rà soát các cách viết trạng thái
4,InventoryBalances -> InventoryMovements,Shared key coverage,SKU_ID,25 giá trị không tìm thấy ở bảng phải,25,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...
5,InventoryBalances -> ProductVariants,Shared key coverage,SKU_ID,0 giá trị không tìm thấy ở bảng phải,0,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...
6,InventoryBalances -> PurchaseOrderLines,Shared key coverage,SKU_ID,27 giá trị không tìm thấy ở bảng phải,27,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...
7,InventoryBalances -> SalesOrderLines,Shared key coverage,SKU_ID,31 giá trị không tìm thấy ở bảng phải,31,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...
8,InventoryBalances -> SKUSuppliers,Shared key coverage,SKU_ID,0 giá trị không tìm thấy ở bảng phải,0,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...
9,InventoryMovements -> ProductVariants,Shared key coverage,SKU_ID,0 giá trị không tìm thấy ở bảng phải,0,Khóa đối chiếu: SKU_ID = SKU_ID; cần xác nhận ...


Cell 6 – Phân tích tuần không có giao dịch theo SKU

In [8]:
weekly_gap_rows = []
weekly_gap_detail = []

for table_name, df in tables.items():

    date_col = find_column(df, CANDIDATES["date"])
    sku_col = find_column(df, ["SKU", "SKUCode", "ProductSKU", "VariantSKU"])
    qty_col = find_column(df, CANDIDATES["quantity"])

    if not (date_col and sku_col and qty_col):
        continue

    temp = df[[date_col, sku_col, qty_col]].copy()

    temp[date_col] = pd.to_datetime(
        temp[date_col],
        errors="coerce"
    )
    temp[qty_col] = pd.to_numeric(
        temp[qty_col],
        errors="coerce"
    )

    temp = temp.dropna(subset=[date_col, sku_col])

    if temp.empty:
        continue

    temp["week"] = (
        temp[date_col]
        .dt.to_period("W")
        .dt.start_time
    )

    observed = (
        temp.groupby([sku_col, "week"])
        .agg(
            transaction_rows=(qty_col, "size"),
            total_quantity=(qty_col, "sum")
        )
        .reset_index()
    )

    all_weeks = pd.date_range(
        temp["week"].min(),
        temp["week"].max(),
        freq="W-MON"
    )

    all_skus = temp[sku_col].dropna().unique()

    full_index = pd.MultiIndex.from_product(
        [all_skus, all_weeks],
        names=[sku_col, "week"]
    )

    complete_grid = (
        observed.set_index([sku_col, "week"])
        .reindex(full_index)
        .reset_index()
    )

    complete_grid["no_observed_transaction"] = (
        complete_grid["transaction_rows"].isna()
    )

    for sku, group in complete_grid.groupby(sku_col):
        gap_weeks = group.loc[
            group["no_observed_transaction"], "week"
        ].sort_values()

        # Đếm chuỗi tuần liên tiếp không có giao dịch quan sát được.
        longest_gap = 0
        current_gap = 0
        previous_week = None

        for week in gap_weeks:
            if (
                previous_week is not None
                and week - previous_week == pd.Timedelta(days=7)
            ):
                current_gap += 1
            else:
                current_gap = 1

            longest_gap = max(longest_gap, current_gap)
            previous_week = week

        weekly_gap_rows.append({
            "table": table_name,
            "sku": sku,
            "total_weeks_in_period": len(group),
            "weeks_with_observed_transaction": int(
                (~group["no_observed_transaction"]).sum()
            ),
            "weeks_without_observed_transaction": int(
                group["no_observed_transaction"].sum()
            ),
            "no_transaction_rate": float(
                group["no_observed_transaction"].mean()
            ),
            "longest_consecutive_gap_weeks": longest_gap
        })

    complete_grid["table"] = table_name
    weekly_gap_detail.append(complete_grid)


weekly_gap_df = pd.DataFrame(weekly_gap_rows)

weekly_gap_detail_df = (
    pd.concat(weekly_gap_detail, ignore_index=True)
    if weekly_gap_detail else pd.DataFrame()
)

display(weekly_gap_df.head(30))

if weekly_gap_df.empty:
    print("Không tìm được bảng có đủ Date, SKU và Quantity.")

,table,sku,total_weeks_in_period,weeks_with_observed_transaction,weeks_without_observed_transaction,no_transaction_rate,longest_consecutive_gap_weeks
0,InventoryMovements,SKU0000001,132,127,5,0.037879,5
1,InventoryMovements,SKU0000002,132,127,5,0.037879,5
2,InventoryMovements,SKU0000003,132,127,5,0.037879,5
3,InventoryMovements,SKU0000004,132,132,0,0.000000,0
4,InventoryMovements,SKU0000005,132,127,5,0.037879,1
5,InventoryMovements,SKU0000006,132,118,14,0.106061,4
6,InventoryMovements,SKU0000007,132,121,11,0.083333,11
7,InventoryMovements,SKU0000008,132,121,11,0.083333,11
8,InventoryMovements,SKU0000009,132,54,78,0.590909,11
9,InventoryMovements,SKU0000010,132,113,19,0.143939,19


Cell 7 – Tổng hợp và xuất data_quality_report.xlsx

In [9]:
quality_report_path = REPORT_DIR / "data_quality_report.xlsx"

with pd.ExcelWriter(
    quality_report_path,
    engine="openpyxl"
) as writer:

    dimension_df.to_excel(
        writer,
        sheet_name="Dimension Scores",
        index=False
    )

    issues_df.to_excel(
        writer,
        sheet_name="Data Issues",
        index=False
    )

    timeliness_df.to_excel(
        writer,
        sheet_name="Timeliness",
        index=False
    )

    consistency_df.to_excel(
        writer,
        sheet_name="Consistency",
        index=False
    )

    weekly_gap_df.to_excel(
        writer,
        sheet_name="SKU Weekly Gaps",
        index=False
    )

    weekly_gap_detail_df.to_excel(
        writer,
        sheet_name="Weekly Gap Detail",
        index=False
    )

print("Đã xuất:", quality_report_path)

Đã xuất: C:\Users\admin\Downloads\important\SKU-Weekly-Sales-Forecasting\reports\data_quality_report.xlsx
